<a href="https://www.kaggle.com/code/saibhossain/improved-naive-rag?scriptVersionId=309438113" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# What this improved naive RAG includes

Still naive RAG, but cleaner:

1. Load multiple PDFs from folder
2. Extract text page by page
3. Split into chunks
4. Build embeddings
5. Store in FAISS
6. Retrieve top-k chunks
7. Build grounded prompt
8. Generate answer
9. Show sources and evidence

## Load dataset 

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Setup

In [ ]:
!pip -q install langchain langchain-huggingface faiss-cpu 
!pip -q install sentence-transformers pypdf
!pip install langchain-community langchain_core
!pip -q install langchain-google-genai

## Imports

In [ ]:
import os
import glob
from dataclasses import dataclass
from typing import List, Dict, Any

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document


from langchain_google_genai import ChatGoogleGenerativeAI

## Load API key

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GOOGLE_API_KEY")

## Create config

In [ ]:
from dataclasses import dataclass

@dataclass
class RAGConfig:
    pdf_dir :str = "/kaggle/input/datasets/saibhossain/rag-practice"
    embedding_model_name:str = "sentence-transformers/all-MiniLM-L6-v2"
    llm_model_name:str = "gemini-2.5-flash"
    chunk_size: int = 800
    chunk_overlap: int = 150
    top_k : int= 4
    temperature: float = 0.2

config = RAGConfig()
print(config)

# Step 1: Load PDF

In [ ]:
def get_pdf_paths(pdf_dir: str) -> List[str]:
    pdf_paths = glob.glob(os.path.join(pdf_dir, "*.pdf"))
    pdf_paths = sorted(pdf_paths)
    return pdf_paths

def load_pdfs_from_folder(pdf_dir: str) -> List[Document]:
    pdf_paths = get_pdf_paths(pdf_dir)
    
    if not pdf_paths:
        raise FileNotFoundError(f"No PDF files found in: {pdf_dir}")
    
    all_docs = []
    
    for pdf_path in pdf_paths:
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()
        
        # Add cleaner metadata
        for doc in docs:
            doc.metadata["file_name"] = os.path.basename(pdf_path)
        
        all_docs.extend(docs)
    
    return all_docs

documents = load_pdfs_from_folder(config.pdf_dir)

print(f"Total pages loaded: {len(documents)}")
print("\nSample metadata:")
print(documents[0].metadata)

print("\nSample page text preview:")
print(documents[0].page_content[:1000])

# Step 2: Chunk documents

In [ ]:
# split documents into chunks
def chunk_documents(documents:List[Document], chunk_size:int, chunk_overlap:int )-> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(documents)
    return chunks
    
# Run chunking

chunks = chunk_documents( documents=documents,chunk_size=config.chunk_size,chunk_overlap=config.chunk_overlap)

print(f"Total chunks created: {len(chunks)}")

print("\nExample chunk metadata:")
print(chunks[0].metadata)

print("\nExample chunk text:")
print(chunks[0].page_content[:1000])

# Step 3: Create embedding model

In [ ]:
def build_embedding_model(model_name: str):
    embedding_model = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embedding_model

embedding_model = build_embedding_model(config.embedding_model_name)
print("Embedding model loaded successfully")

In [ ]:
# Build FAISS vector store

def build_vectorstore(chunks: List[Document], embedding_model) -> FAISS:
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )
    return vectorstore

vectorstore = build_vectorstore(chunks, embedding_model)
print("FAISS vector store created successfully")

# Step 4: Retriever

In [ ]:
def build_retriever(vectorstore: FAISS, top_k: int = 4):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": top_k}
    )
    return retriever

retriever = build_retriever(vectorstore, config.top_k)
print("Retriever created successfully")

In [ ]:
# Build LLM

def build_llm(model_name: str, temperature: float):
    llm = ChatGoogleGenerativeAI(
        model=model_name,
        temperature=temperature
    )
    return llm

llm = build_llm(config.llm_model_name, config.temperature)
print("LLM initialized successfully")

In [ ]:
# format retreved context 

def format_context(retrieved_docs: List[Document]) -> str:
    context_parts = []
    
    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("file_name", "unknown_file")
        page = doc.metadata.get("page", "unknown_page")
        text = doc.page_content.strip()
        
        context_parts.append(
            f"[Chunk {i} | Source: {source} | Page: {page}]\n{text}"
        )
    
    return "\n\n".join(context_parts)

In [ ]:
# Build RAG prompt

def build_rag_prompt(query: str, retrieved_docs: List[Document]) -> str:
    context = format_context(retrieved_docs)
    
    prompt = f"""
You are a helpful personal AI assistant.

Your job is to answer the user's question using ONLY the retrieved context below.

Rules:
1. Use only the provided context.
2. Do not add outside knowledge.
3. Do not use your pre train knowledge.
4. If the answer is not clearly present in the context, say:
   "I could not find the answer in the provided context."
5. When possible, mention the source file name.

Retrieved Context:
{context}

User Question:
{query}

Answer:
"""
    return prompt.strip()

In [ ]:
# Retrieval-only inspection function

def inspect_retrieval(query: str, retriever) -> List[Document]:
    retrieved_docs = retriever.invoke(query)
    
    print(f"Query: {query}")
    print(f"Retrieved {len(retrieved_docs)} chunks\n")
    
    for i, doc in enumerate(retrieved_docs, start=1):
        print(f"{'='*80}")
        print(f"Chunk {i}")
        print(f"Source File: {doc.metadata.get('file_name', 'unknown')}")
        print(f"Page: {doc.metadata.get('page', 'unknown')}")
        print("-"*80)
        print(doc.page_content[:1500])
        print("\n")
    
    return retrieved_docs

In [ ]:
def ask_naive_rag(query: str, retriever, llm) -> Dict[str, Any]:
    retrieved_docs = retriever.invoke(query)
    prompt = build_rag_prompt(query, retrieved_docs)
    response = llm.invoke(prompt)
    
    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt,
        "answer": response.content
    }

def print_rag_result(result: Dict[str, Any]):
    print("="*100)
    print("QUERY")
    print("="*100)
    print(result["query"])
    
    print("\n" + "="*100)
    print("ANSWER")
    print("="*100)
    print(result["answer"])
    
    print("\n" + "="*100)
    print("RETRIEVED SOURCES")
    print("="*100)
    
    for i, doc in enumerate(result["retrieved_docs"], start=1):
        print(f"\nChunk {i}")
        print(f"File: {doc.metadata.get('file_name', 'unknown')}")
        print(f"Page: {doc.metadata.get('page', 'unknown')}")
        print(f"Preview: {doc.page_content[:500]}")

### Test retrieval first

In [ ]:
test_query = "What is the main topic discussed in these documents?"

retrieved_docs = inspect_retrieval(test_query, retriever)

In [ ]:
result = ask_naive_rag(
    query="Summarize the key idea from the documents.",
    retriever=retriever,
    llm=llm
)

print_rag_result(result)


### Ask more targeted questions

In [ ]:
queries = [
    "What problem does the paper try to solve?",
    "What dataset was used in Lung Cancer Detection paper?",
    "Compare two papers ",
    "Table values",
    "What methods are used?",
    "What are the limitations mentioned in the documents?"
]

for q in queries:
    print("\n\n" + "#" * 120)
    result = ask_naive_rag(query=q, retriever=retriever, llm=llm)
    print_rag_result(result)

---

| Query                                | Retrieved Correct? | Answer Correct? | Failure Type      | Root Cause     |
| ------------------------------------ | ------------------ | --------------- | ----------------- | -------------- |
| What problem does the paper solve?   | ❌                  | ❌               | Ambiguity         | multiple PDFs  |
| What dataset was used in Lung Cancer Detection paper? | ✅                  | ✅               | -                 | good retrieval |
| Compare two papers                   | ❌                  | ❌               | reasoning failure | no structure   |
| Table values                         | ❌                  | ❌               | parsing failure   | PDF issue      |
| What methods are used?                        | ❌                  | ❌               | parsing failure   | PDF issue      |
| What are the limitations mentioned in the documents?                        | ❌                  | ❌               | parsing failure   | PDF issue      |


# 👨‍💻 Author
# **Md Saib Hossain**
**AI Engineer • AI / ML / LLM & AI Safety Researcher**  
**Agentic AI Developer • Researcher in Autonomous & Multi-Agent Systems • Advanced Agentic AI Architect**

Designing safe, scalable, and human-centered intelligent systems for real-world healthcare and autonomous AI applications.

<p align="left">
  <a href="mailto:saibhossain5@gmail.com">
    <img src="https://img.shields.io/badge/Email-saibhossain5%40gmail.com-red?style=flat&logo=gmail">
  </a>
  <a href="https://saibhossain.github.io/">
    <img src="https://img.shields.io/badge/Portfolio-Visit-blue?style=flat&logo=google-chrome">
  </a>
  <a href="https://github.com/Saibhossain">
    <img src="https://img.shields.io/badge/GitHub-Profile-black?style=flat&logo=github">
  </a>
  <a href="https://linkedin.com/in/saib-hossain-182834229">
    <img src="https://img.shields.io/badge/LinkedIn-Connect-0A66C2?style=flat&logo=linkedin">
  </a>
</p>
